# Problem Statement

Customer churn is a critical challenge for telecom companies, as losing customers directly impacts revenue and growth. The objective of this project is to develop a machine learning system that can predict whether a customer is likely to churn (leave the service) based on historical customer data such as usage patterns, service details, and account information.

By accurately identifying potential churners, businesses can take proactive measures (e.g., targeted offers, improved services) to retain customers and improve overall profitability.

In [1]:
pip install pandas numpy scikit-learn joblib

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Here I import all essential libraries that are used in this project
import pandas as pd
import numpy as np
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
import joblib

# Dataset Loading and Preprocessing

The dataset used in this project consists of two parts:

Training dataset: churn-bigml-80.csv
Testing dataset: churn-bigml-20.csv
## Key preprocessing steps:
Loaded datasets using Pandas
Separated features (X) and target variable (Churn)
Identified:
Numerical features (e.g., call duration, charges)
Categorical features (e.g., state, plan type)
Handled missing values:
Numerical → median imputation
Categorical → most frequent imputation
Applied:
StandardScaler for numerical data
OneHotEncoder for categorical data

All preprocessing steps were combined using ColumnTransformer inside a Pipeline, ensuring consistency and reusability.

In [3]:
# Here I read the dataset as a csv file
train_df = pd.read_csv("churn-bigml-80.csv")
test_df = pd.read_csv("churn-bigml-20.csv")

In [4]:
# Here I use head function to see the first five records
train_df.head()

,State,Account length,Area code,International plan,Voice mail plan,Number vmail messages,Total day minutes,Total day calls,Total day charge,Total eve minutes,Total eve calls,Total eve charge,Total night minutes,Total night calls,Total night charge,Total intl minutes,Total intl calls,Total intl charge,Customer service calls,Churn
0,KS,128,415,No,Yes,25,265.1,110,45.07,197.4,99,16.78,244.7,91,11.01,10.0,3,2.70,1,False
1,OH,107,415,No,Yes,26,161.6,123,27.47,195.5,103,16.62,254.4,103,11.45,13.7,3,3.70,1,False
2,NJ,137,415,No,No,0,243.4,114,41.38,121.2,110,10.30,162.6,104,7.32,12.2,5,3.29,0,False
3,OH,84,408,Yes,No,0,299.4,71,50.90,61.9,88,5.26,196.9,89,8.86,6.6,7,1.78,2,False
4,OK,75,415,Yes,No,0,166.7,113,28.34,148.3,122,12.61,186.9,121,8.41,10.1,3,2.73,3,False


In [5]:
# Here I use tail function to see the last five records
train_df.tail()

,State,Account length,Area code,International plan,Voice mail plan,Number vmail messages,Total day minutes,Total day calls,Total day charge,Total eve minutes,Total eve calls,Total eve charge,Total night minutes,Total night calls,Total night charge,Total intl minutes,Total intl calls,Total intl charge,Customer service calls,Churn
2661,SC,79,415,No,No,0,134.7,98,22.90,189.7,68,16.12,221.4,128,9.96,11.8,5,3.19,2,False
2662,AZ,192,415,No,Yes,36,156.2,77,26.55,215.5,126,18.32,279.1,83,12.56,9.9,6,2.67,2,False
2663,WV,68,415,No,No,0,231.1,57,39.29,153.4,55,13.04,191.3,123,8.61,9.6,4,2.59,3,False
2664,RI,28,510,No,No,0,180.8,109,30.74,288.8,58,24.55,191.9,91,8.64,14.1,6,3.81,2,False
2665,TN,74,415,No,Yes,25,234.4,113,39.85,265.9,82,22.60,241.4,77,10.86,13.7,4,3.70,0,False


In [6]:
# Here I see the shape of the dataset
"Train Data Shape:", train_df.shape

('Train Data Shape:', (2666, 20))

In [7]:
# Here I see the whole information about the dataset
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2666 entries, 0 to 2665
Data columns (total 20 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   State                   2666 non-null   object 
 1   Account length          2666 non-null   int64  
 2   Area code               2666 non-null   int64  
 3   International plan      2666 non-null   object 
 4   Voice mail plan         2666 non-null   object 
 5   Number vmail messages   2666 non-null   int64  
 6   Total day minutes       2666 non-null   float64
 7   Total day calls         2666 non-null   int64  
 8   Total day charge        2666 non-null   float64
 9   Total eve minutes       2666 non-null   float64
 10  Total eve calls         2666 non-null   int64  
 11  Total eve charge        2666 non-null   float64
 12  Total night minutes     2666 non-null   float64
 13  Total night calls       2666 non-null   int64  
 14  Total night charge      2666 non-null   

In [8]:
# Here I see that is there any missing value in this dataset
print(train_df.isnull().sum())

State                     0
Account length            0
Area code                 0
International plan        0
Voice mail plan           0
Number vmail messages     0
Total day minutes         0
Total day calls           0
Total day charge          0
Total eve minutes         0
Total eve calls           0
Total eve charge          0
Total night minutes       0
Total night calls         0
Total night charge        0
Total intl minutes        0
Total intl calls          0
Total intl charge         0
Customer service calls    0
Churn                     0
dtype: int64


In [9]:
print(test_df.isnull().sum())

State                     0
Account length            0
Area code                 0
International plan        0
Voice mail plan           0
Number vmail messages     0
Total day minutes         0
Total day calls           0
Total day charge          0
Total eve minutes         0
Total eve calls           0
Total eve charge          0
Total night minutes       0
Total night calls         0
Total night charge        0
Total intl minutes        0
Total intl calls          0
Total intl charge         0
Customer service calls    0
Churn                     0
dtype: int64


In [10]:
print(train_df["Churn"].value_counts())

Churn
False    2278
True      388
Name: count, dtype: int64


In [11]:
# Here I split the dataset into X and Y parts
X_train = train_df.drop("Churn", axis=1)
y_train = train_df["Churn"]

In [12]:
X_test = test_df.drop("Churn", axis=1)
y_test = test_df["Churn"]

In [13]:
numeric_features = X_train.select_dtypes(include=["int64", "float64"]).columns
categorical_features = X_train.select_dtypes(include=["object"]).columns

In [14]:
# Here I preprocess the data (scalling, encoding, etc) using the pipeline
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

In [15]:
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

In [16]:
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

In [17]:
pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, random_state=42))
])

# Model Development and Training

A complete machine learning pipeline was built using Scikit-learn:

Integrated preprocessing and model training into a single Pipeline
Trained multiple models:
Logistic Regression (baseline model)
Random Forest Classifier (ensemble model)

# Hyperparameter Tuning:
Used GridSearchCV to:
Test multiple model configurations
Select the best model based on performance

This ensures the model is optimized and generalizable.

In [18]:
# Here I use hyperparameter tuning using GridSearchCV
param_grid = [
    {
        "classifier": [LogisticRegression(max_iter=1000, random_state=42)],
        "classifier__C": [0.1, 1, 10]
    },
    {
        "classifier": [RandomForestClassifier(random_state=42)],
        "classifier__n_estimators": [100, 200],
        "classifier__max_depth": [None, 10, 20]
    }
]
grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    verbose=1
)

In [19]:
# Here I train the model
grid_search.fit(X_train, y_train)
print("\nBest Parameters:", grid_search.best_params_)

Fitting 5 folds for each of 9 candidates, totalling 45 fits

Best Parameters: {'classifier': RandomForestClassifier(random_state=42), 'classifier__max_depth': 20, 'classifier__n_estimators': 200}


# Evaluation with Relevant Metrics

The trained model was evaluated on the test dataset using:

Accuracy Score → Overall correctness of predictions
Classification Report → Includes:
Precision
Recall
F1-score
Confusion Matrix → Shows:
True Positives, False Positives, etc.
ROC-AUC Score → Measures model’s ability to distinguish between churn and non-churn

These metrics provide a comprehensive understanding of model performance, especially important for imbalanced datasets like churn prediction.

In [20]:
# Here I evaluate the performance of the model 
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
print("\nAccuracy:", accuracy_score(y_test, y_pred))


Accuracy: 0.9475262368815592


In [21]:
print("\nClassification Report:\n", classification_report(y_test, y_pred))


Classification Report:
               precision    recall  f1-score   support

       False       0.95      1.00      0.97       572
        True       0.97      0.65      0.78        95

    accuracy                           0.95       667
   macro avg       0.96      0.82      0.88       667
weighted avg       0.95      0.95      0.94       667



In [22]:
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

Confusion Matrix:
 [[570   2]
 [ 33  62]]


In [23]:
print("ROC-AUC Score:", roc_auc_score(y_test, y_pred))

ROC-AUC Score: 0.8245675377254325


In [24]:
joblib.dump(best_model, "churn_pipeline.pkl")
print("\n Model saved as churn_pipeline.pkl")


 Model saved as churn_pipeline.pkl


# Final Summary / Insights
A fully automated end-to-end ML pipeline was successfully developed
The pipeline handles:
Data preprocessing
Model training
Hyperparameter tuning
Evaluation
The best-performing model was selected using GridSearchCV
The final pipeline was exported using joblib, making it:
Reusable
Ready for deployment